In [10]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

import os

# HeartLab

In [11]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

import os

In [38]:
data_dir = 'data/dataset'

msmt_defs = pd.read_csv(os.path.join(data_dir,'MEASUREMENT_DEFINITIONS.csv'))
msmt_formula = pd.read_csv(os.path.join(data_dir,'MEASUREMENT_FORMULA.csv'))
msmt_groups = pd.read_csv(os.path.join(data_dir,'MEASUREMENT_GROUPS.csv'))
msmt_intersects = pd.read_csv(os.path.join(data_dir,'MEASUREMENT_INTERSECTS.csv'))
msmt_lists = pd.read_csv(os.path.join(data_dir,'MEASUREMENT_LISTS.csv'))
msmt = pd.read_csv(os.path.join(data_dir,'MEASUREMENTS.csv'))

findings = pd.read_csv(os.path.join(data_dir,'ENCOADMIN_FINDINGS.csv'))
finding_groups = pd.read_csv(os.path.join(data_dir,'ENCOADMIN_FINDING_GROUPS.csv'))
finding_intersects = pd.read_csv(os.path.join(data_dir,'ENCOADMIN_FINDING_INTERSECTS.csv'))

patients = pd.read_csv(os.path.join(data_dir,'Patients_No_PHI.csv'))
reports = pd.read_csv(os.path.join(data_dir,'REPORTS.csv'))
series = pd.read_csv(os.path.join(data_dir,'SERIES.csv'))
studies = pd.read_csv(os.path.join(data_dir,'STUDIES.csv'))

/tmp/ipykernel_21775/2801898404.py:6: DtypeWarning: Columns (7,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  msmt_intersects = pd.read_csv(os.path.join(data_dir,'MEASUREMENT_INTERSECTS.csv'))
/tmp/ipykernel_21775/2801898404.py:15: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  reports = pd.read_csv(os.path.join(data_dir,'REPORTS.csv'))
/tmp/ipykernel_21775/2801898404.py:16: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  series = pd.read_csv(os.path.join(data_dir,'SERIES.csv'))
/tmp/ipykernel_21775/2801898404.py:17: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  studies = pd.read_csv(os.path.join(data_dir,'STUDIES.csv'))


In [39]:
hl_deid = pd.read_csv('data/dataset/echo-study-deid.csv')

In [40]:
# ensure the join keys have the same dtype
series['STUD_ID']  = series['STUD_ID'].astype(str)
studies['ID']      = studies['ID'].astype(str)

# build the 2-column map
study_series_map = (
    series[['ID', 'STUD_ID']]
        .merge(
            studies[['ID']].rename(columns={'ID': 'STUDY_ID'}),   # avoid _x/_y suffixes
            left_on='STUD_ID',
            right_on='STUDY_ID',
            how='left'
        )
        .rename(columns={'ID': 'SERI_ID'})                        # series ID → SERI_ID
        [['SERI_ID', 'STUDY_ID']]
)

# stats
overlap_count = study_series_map['STUDY_ID'].notna().sum()
dropped_count = study_series_map['STUDY_ID'].isna().sum()

print(f'overlaps: {overlap_count}, dropped: {dropped_count}')


overlaps: 1028007, dropped: 0


In [42]:
study_series_map.head()

,SERI_ID,STUDY_ID
0,442524,227777
1,442525,227774
2,442537,227784
3,442539,227786
4,442540,227787


Basically getting the OriginalID from studies, which called `STUDY_INSTANCE_UID`. We are also getting `STUDY_DATE` so we can create temporal splits.

In [43]:
# ── dtype alignment ────────────────────────────────────────────────────
study_series_map['SERI_ID'] = study_series_map['SERI_ID'].astype(str)
reports['SERI_ID']          = reports['SERI_ID'].astype(str)
studies['ID']               = studies['ID'].astype(str)           # already str but keep consistent

# ── add STUDY_INSTANCE_UID from studies ───────────────────────────────
study_series_map = (
    study_series_map
        .merge(
            studies[['ID', 'STUDY_INSTANCE_UID', 'STUDY_DATE']].rename(columns={'ID': 'STUDY_ID'}),
            on='STUDY_ID',
            how='left'
        )
)

# ── keep / order desired columns ──────────────────────────────────────
study_series_map = study_series_map[['SERI_ID', 'STUDY_ID', 'STUDY_INSTANCE_UID', 'STUDY_DATE']]

# ── stats ─────────────────────────────────────────────────────────────
uid_overlap   = study_series_map['STUDY_INSTANCE_UID'].notna().sum()
uid_dropped   = study_series_map['STUDY_INSTANCE_UID'].isna().sum()

print(f'UID    overlaps: {uid_overlap}, dropped: {uid_dropped}')


UID    overlaps: 1028007, dropped: 0


In [44]:
study_series_map.head()

,SERI_ID,STUDY_ID,STUDY_INSTANCE_UID,STUDY_DATE
0,442524,227777,1.2.840.113680.1.103.60427.1265116737.713938,02/02/10 08:18:57
1,442525,227774,1.2.840.113680.1.103.57589.1265126889.672993,02/02/10 08:08:09
2,442537,227784,IMAGELESS.STUDY.UID.227784,02/02/10 07:14:31
3,442539,227786,IMAGELESS.STUDY.UID.227786,02/01/10 13:51:24
4,442540,227787,IMAGELESS.STUDY.UID.227787,02/01/10 11:25:25


Here, we are trying to get the `REP_ID` from the reports table, since that's what the finding codes map to. There are 318K reports and 1.02M series, so most of them will inevitably be dropped (findings are at the report level).

In [60]:
# normalise keys ───────────────────────────────────────────────────────
series['ID']        = series['ID'].astype(str)                 # series PK
series['STUD_ID']   = series['STUD_ID'].astype(str)            # FK to studies
reports['SERI_ID']  = (reports['SERI_ID']
                         .astype(str)
                         .str.replace(r'\.0$', '', regex=True))# drop ".0"
studies['ID']       = studies['ID'].astype(str)                # studies PK

# build map ────────────────────────────────────────────────────────────
study_series_map = (
    # start from series → rename ID → SERI_ID
    series[['ID', 'STUD_ID']].rename(columns={'ID': 'SERI_ID'})
        # add REP_ID from reports via SERI_ID
        .merge(
            reports[['SERI_ID', 'ID']].rename(columns={'ID': 'REP_ID'}),
            on='SERI_ID', how='left'
        )
        # add STUDY_ID + STUDY_INSTANCE_UID (match STUD_ID ↔ ID)
        .merge(
            studies[['ID', 'STUDY_INSTANCE_UID', 'STUDY_DATE']]
                   .rename(columns={'ID': 'STUDY_ID'}),
            left_on='STUD_ID', right_on='STUDY_ID', how='left'
        )
        # final column order
        [['REP_ID', 'STUDY_DATE', 'STUDY_ID', 'SERI_ID', 'STUDY_INSTANCE_UID']]
)

# stats ────────────────────────────────────────────────────────────────
rep_overlap = study_series_map['REP_ID'].notna().sum()
rep_missing = study_series_map['REP_ID'].isna().sum()
print(f'report overlaps: {rep_overlap}, dropped: {rep_missing}')

study_series_map = study_series_map.dropna(subset=['REP_ID']).reset_index(drop=True)

report overlaps: 316139, dropped: 712201


In [72]:
print(len(study_series_map))
study_series_map.head()

316139


,REP_ID,STUDY_DATE,STUDY_ID,SERI_ID,STUDY_INSTANCE_UID
0,218588.0,02/02/10 08:08:09,227774,442525,1.2.840.113680.1.103.57589.1265126889.672993
1,218592.0,02/02/10 07:14:31,227784,442537,IMAGELESS.STUDY.UID.227784
2,218594.0,02/01/10 13:51:24,227786,442539,IMAGELESS.STUDY.UID.227786
3,218595.0,02/01/10 11:25:25,227787,442540,IMAGELESS.STUDY.UID.227787
4,218598.0,02/02/10 08:05:34,227773,442543,1.2.840.113663.1500.1.278589547.1.1.20100202.80535.31


In [73]:
print(len(hl_deid))
hl_deid.head()

237022


,OriginalStudyID,OriginalPatientID,DeidentifiedStudyID,DeidentifiedPatientID
0,1.2.840.113543.6.6.2.0.97083308851.20060302142809,3266450,1.2.276.0.7230010.3.1.2.1714578744.1.1703334530.11867061,00877fe5-b4ab-4df8-b6f1-68b947cec98d
1,1.2.840.113619.2.185.2475.1245236079.0.2,2267861,1.2.276.0.7230010.3.1.2.811753780.1.1703348281.12273871,d0fc0c0a-77d9-443a-8b3c-d896acac82cb
2,1.2.840.113619.2.98.4671.1161587002.0.1567,3376759,1.2.276.0.7230010.3.1.2.811753780.1.1703377162.13055155,cca0fe70-43cf-4ad4-b9cc-3e267c3566de
3,1.2.840.113680.1.103.65868.1148668030.13782,3289867,1.2.276.0.7230010.3.1.2.859333938.1.1703688876.16737956,85fbd721-2c36-46c6-a0c6-945d99945763
4,1.3.12.2.1107.5.8.9.10001229899769.20170330135638788,3716998,1.2.276.0.7230010.3.1.2.845494328.1.1703760802.23327962,648832e3-b2a9-46d2-b814-ae2fcdf20d11


Now, we map `STUDY_INSTANCE_UID` to `OriginalStudyID` so we can map to the videos which are named according to `DeidentifiedStudyID`. We use the de-identification key given to us by Joe.

In [74]:
study_series_unique = study_series_map.drop_duplicates(subset=["SERI_ID"])

hl_deid_merged = hl_deid.merge(
    study_series_unique,
    left_on="OriginalStudyID",
    right_on="STUDY_INSTANCE_UID",
    how="left"
)

The result will be the subset (237K) of the superset (316K), which is the set of all deid keys. Some of the deid keys will also not be in the superset, so the REP_IDs will be NaN.

In [76]:
print(len(hl_deid_merged))
hl_deid_merged.head()

237199


,OriginalStudyID,OriginalPatientID,DeidentifiedStudyID,DeidentifiedPatientID,REP_ID,STUDY_DATE,STUDY_ID,SERI_ID,STUDY_INSTANCE_UID
0,1.2.840.113543.6.6.2.0.97083308851.20060302142809,3266450,1.2.276.0.7230010.3.1.2.1714578744.1.1703334530.11867061,00877fe5-b4ab-4df8-b6f1-68b947cec98d,81653.0,03/02/06 14:47:59,85175,167669,1.2.840.113543.6.6.2.0.97083308851.20060302142809
1,1.2.840.113619.2.185.2475.1245236079.0.2,2267861,1.2.276.0.7230010.3.1.2.811753780.1.1703348281.12273871,d0fc0c0a-77d9-443a-8b3c-d896acac82cb,198820.0,06/17/09 11:54:39,203384,397155,1.2.840.113619.2.185.2475.1245236079.0.2
2,1.2.840.113619.2.98.4671.1161587002.0.1567,3376759,1.2.276.0.7230010.3.1.2.811753780.1.1703377162.13055155,cca0fe70-43cf-4ad4-b9cc-3e267c3566de,109091.0,10/23/06 10:45:50,104164,226253,1.2.840.113619.2.98.4671.1161587002.0.1567
3,1.2.840.113680.1.103.65868.1148668030.13782,3289867,1.2.276.0.7230010.3.1.2.859333938.1.1703688876.16737956,85fbd721-2c36-46c6-a0c6-945d99945763,91632.0,05/26/06 13:27:10,92130,189093,1.2.840.113680.1.103.65868.1148668030.13782
4,1.3.12.2.1107.5.8.9.10001229899769.20170330135638788,3716998,1.2.276.0.7230010.3.1.2.845494328.1.1703760802.23327962,648832e3-b2a9-46d2-b814-ae2fcdf20d11,NaN,NaN,NaN,NaN,NaN


For each `STUDY_INSTANCE_UID`, keep one row—prefer the one with a non-null `REP_ID`.

In [77]:
out = (
    hl_deid_merged
        .assign(_keep=hl_deid_merged['REP_ID'].notna())   # True when REP_ID present
        .sort_values('_keep', ascending=False)            # rows with REP_ID first
        .drop_duplicates('STUDY_INSTANCE_UID', keep='first')
        .drop(columns='_keep')
)


In [78]:
len(out)

196448

In [79]:
out = out.dropna(subset=['REP_ID']).reset_index(drop=True)

In [80]:
len(out)

196447

In [81]:
out.to_csv('data/dataset/heartlab_rep_study_video.csv', index=False)